# Perch V2 — Optimal Confidence Threshold

Perch gives every detection a confidence score — but *how high* does a score have to
be before you trust it? That cutoff is species-specific: a 0.3 for one bird can be as
reliable as a 0.7 for another.

This notebook estimates, **per species**, the confidence threshold at which a detection
has a chosen probability (default **95 %**) of being correct. It is the companion to
`species_ID.ipynb`: that notebook finds detections and cuts clips, you listen to the
clips and sort them, and this one turns your verdicts into a threshold.

## Workflow

1. **Download** Perch V2 from Kaggle with `kagglehub` (cached after the first run).
2. **Load** the TensorFlow SavedModel and its bundled list of species labels.
3. Point it at your **human-validated** clips — the ones you listened to and sorted
   into `correct/` and `incorrect/` folders.
4. **Recover each clip's confidence** — by default straight from the
   `all_detections.csv` that produced it, so the number being calibrated is exactly the
   number `species_ID.ipynb` computed. Clips from elsewhere can be **re-scored** instead
   (`CONFIDENCE_SOURCE = "rescore"` in §5), which runs the full pipeline: load & resample
   to mono 32 kHz (`polyphase`, as hoplite does) → cut into 5-second windows →
   peak-normalize → run the model → keep the target species' strongest window.
5. **Fit** a logistic regression of correctness on the logit-transformed confidence,
   per species, and **solve** for the confidence at your target probability.
6. **Save** a thresholds table, a precision-by-confidence-band table, a plot, and a
   `report.md`.

## Method

Wood et al. 2023; validated against Bota et al. 2023.

1. Back-transform each confidence to the logit scale `L = ln(c / (1 − c))`.
2. Fit `P(correct) = sigmoid(b0 + b1·L)` by maximum likelihood.
3. Invert it for the confidence `c*` where `P(correct) = target`:
   `L* = (logit(target) − b0) / b1`, then `c* = sigmoid(L*)`.

### Considerations

- **You need validated clips first.** This notebook does not find birds; it calibrates
  the scores of detections a human has already judged. Produce them with `species_ID.ipynb`
  (its §8 segment extraction), listen to them, and sort them into `correct/` and
  `incorrect/`.

- **Preprocessing must match the notebook that produced the clips.** Perch V2 is **not**
  amplitude-invariant, so the same two knobs that move the scores in `species_ID.ipynb`
  move them here — and a threshold fitted on one pipeline does not transfer to another.
  Both are defaults of `perch-hoplite`, and both are set to the *same values* as in
  `species_ID.ipynb`:
  - **Peak normalization** (`TARGET_PEAK = 0.25`) — each 5-second window has its DC offset
    removed and its peak scaled to 0.25.
  - **Resampler** (`RESAMPLER = "polyphase"`) — hoplite resamples with `polyphase`;
    `librosa.load`'s own default is `soxr_hq`. This only matters when the source is not
    already 32 kHz.

  If you change one there, change it here too. The `manifest.json` written in §9 records
  what this run used, so a threshold can always be traced back to the pipeline it fits.

  The surest way to avoid the mismatch entirely is to not re-derive the score at all:
  with the default `CONFIDENCE_SOURCE = "detections_csv"` (§5) this notebook reads the
  confidence `species_ID.ipynb` already wrote, and the preprocessing knobs above stop
  mattering here — they only apply on the `"rescore"` path. That is the default for a
  reason: cutting a clip re-loads the audio, and a clip cut with a different resampler
  than the one used to score it will re-score into a slightly different domain than the
  detections your threshold will be applied to.

- **Sample size matters.** A logistic fit on a handful of clips is meaningless. Aim for
  a few dozen clips per species, spread across the confidence range and including both
  correct and incorrect examples. `species_ID.ipynb`'s bin-balanced sampling (`BIN_WIDTH`,
  `MAX_PER_BIN`) exists to give you exactly that spread.

## Before you run

The repo of this project (**Perch_notebooks**) already contains everything this notebook
needs. If you have already run `species_ID.ipynb`, you are set up.

### 1 · The environment

The Python environment lives in `.venv/`, built with [uv](https://docs.astral.sh/uv/)
from `pyproject.toml`: **Python 3.11** with TensorFlow 2.21, librosa, kagglehub,
soundfile, numpy, pandas, scikit-learn, matplotlib and Jupyter. It is already installed
but, to rebuild it from scratch if needed:

```bash
cd ~/projects/Perch_notebooks
uv sync
```

Pick the right kernel. Top-right of this notebook → *Select Kernel* →
**Perch Notebooks (3.11)** (or `.venv/bin/python`). If the import cell below raises
`ModuleNotFoundError: No module named 'tensorflow'`, you are on the wrong kernel.

### 2 · Audio codecs (optional)

`.wav`, `.flac` and `.ogg` work out of the box: `libsndfile` ships inside the
`soundfile` wheel. Only `.mp3` / `.m4a` need a system decoder:

```bash
sudo apt-get install -y ffmpeg
```

### 3 · A Kaggle account

Perch V2 is downloaded from Kaggle on the first run, cached under
`~/.cache/kagglehub` afterwards, so it only happens once.

1. Accept the model's terms once on the
   [model page](https://www.kaggle.com/models/google/bird-vocalization-classifier).
2. Create a token: Kaggle → *Settings* → *API* → *Create New Token*.
3. Export it **before** launching VS Code / Jupyter, so the kernel inherits it:

```bash
export KAGGLE_API_TOKEN=KGAT_xxxxxxxxxxxxxxxxxxxxxxxx
```

A classic `~/.kaggle/kaggle.json`, or `KAGGLE_USERNAME` + `KAGGLE_KEY`, works too.
(Put the `export` in `~/.bashrc` to make it stick across shells.)

### 4 · Your sorted clips

Run `species_ID.ipynb` with `EXTRACT_SEGMENTS = True`. It writes clips to
`data/outputs/<...>/threshold_<t>/segments/<species>/`. Listen to them, then move each
one into a `correct/` or `incorrect/` folder — the layouts are described in §4. Point
`VALID_DIR` at the result. Results are written to `data/outputs/`.

**Copy or move the clips out of `segments/`** rather than sorting them in place: re-running
the extraction step overwrites that folder and would undo your work.

## 0 · Libraries

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # quiet TensorFlow's start-up logs

import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import librosa
import tensorflow as tf
from IPython.display import display
from sklearn.linear_model import LogisticRegression

print("TensorFlow", tf.__version__, "— libraries ready.")

## 1 · Download Perch V2 from Kaggle

`kagglehub` fetches the model the **first** time and caches it under
`~/.cache/kagglehub` for every run afterwards. The first download needs a Kaggle
API token in your environment (create one at Kaggle → *Settings* → *API*, then
`export KAGGLE_API_TOKEN=...` before launching Jupyter).

We use the **`perch_v2_cpu`** build, which runs on any machine. (The plain
`perch_v2` build is compiled for CUDA and will not run on a CPU.)

In [ ]:
import kagglehub

MODEL_SLUG = "google/bird-vocalization-classifier/tensorFlow2/perch_v2_cpu"

model_path = Path(kagglehub.model_download(MODEL_SLUG))
print("Model files at:", model_path)

## 2 · Load the model and the species labels

A Perch SavedModel is called through its `serving_default` signature: you hand it a
batch of 5-second windows shaped `[N, 160000]` (float32) and it returns, among
other things, a `label` array of shape `[N, 14795]` — one raw score (*logit*) per
species, per window.

The species names live in `assets/labels.csv` inside the downloaded model. Its
first line is a namespace tag, so we skip it; the remaining 14,795 names line up
one-to-one with the `label` scores. `CLASS_INDEX` inverts that list, so a species'
column can be looked up by its scientific name — that is how each folder in §4 is
matched to the score it should be calibrated against.

In [ ]:
model = tf.saved_model.load(str(model_path))
infer = model.signatures["serving_default"]   # inputs=[N, 160000] float32  ->  {"label", "embedding", ...}

SAMPLE_RATE    = 32_000                        # Perch's required input rate (Hz)
WINDOW_S       = 5.0                           # Perch's fixed window; do NOT change
WINDOW_SAMPLES = int(WINDOW_S * SAMPLE_RATE)   # 160000 samples per window

# The two preprocessing knobs, identical to species_ID.ipynb. Both are perch-hoplite's
# defaults (see the notes at the top): they are choices, not fixed requirements, and they
# are why two tools running "the same" model can disagree on confidence. A threshold is
# only valid for the pipeline it was fitted on, so if you changed either of these when
# producing the clips, change it here to match.
RESAMPLER   = "polyphase"   # hoplite's resampler. librosa.load's own default is "soxr_hq"
TARGET_PEAK = 0.25          # hoplite's per-window peak normalization. None disables it

label_lines = (model_path / "assets" / "labels.csv").read_text().splitlines()
CLASS_NAMES = [ln.strip() for ln in label_lines[1:] if ln.strip()]   # skip the header line
CLASS_INDEX = {name: i for i, name in enumerate(CLASS_NAMES)}        # scientific name -> column
print(f"Loaded {len(CLASS_NAMES)} species classes. First few: {CLASS_NAMES[:3]}")

## 3 · Preprocessing

The same three functions as `species_ID.ipynb`, unchanged — deliberately, since the
scores fitted here must be produced the same way as the scores the threshold will be
applied to.

- **`load_audio`** — read any common audio file as mono and resample to 32 kHz with
  `RESAMPLER`. Files already at 32 kHz are unaffected by the choice.
- **`frame_windows`** — slice the clip into 5-second windows, stepping forward by
  `hop_s` seconds each time. Clips shorter than 5 s are padded; a leftover tail
  shorter than one window is dropped.
- **`peak_normalize`** — for each window, subtract the mean (remove DC offset) and
  scale so the loudest sample sits at `TARGET_PEAK`. Setting `TARGET_PEAK = None`
  skips it (like hoplite); setting it to `0` would multiply the audio by zero and
  feed the model **silence** — it would not fail, it would just score nothing.

In [ ]:
def load_audio(path):
    """Load an audio file as mono float32 at Perch's 32 kHz rate.

    RESAMPLER is passed explicitly: librosa.load defaults to "soxr_hq", while
    perch-hoplite resamples with "polyphase". On 24 kHz input the two disagree by
    ~0.03 mean |delta| on the top confidence, so leaving it implicit silently
    decides whether this notebook reproduces hoplite or not.
    """
    audio, _ = librosa.load(str(path), sr=SAMPLE_RATE, mono=True, res_type=RESAMPLER)
    return audio.astype(np.float32)


def frame_windows(audio, hop_s):
    """Cut audio into Perch's 5 s windows, stepping by hop_s seconds.

    Returns an array of shape [n_windows, 160000].
    """
    hop_samples = int(hop_s * SAMPLE_RATE)
    if len(audio) < WINDOW_SAMPLES:
        audio = librosa.util.pad_center(audio, size=WINDOW_SAMPLES, axis=-1)
    frames = librosa.util.frame(
        audio, frame_length=WINDOW_SAMPLES, hop_length=hop_samples, axis=-1
    ).swapaxes(-1, -2)
    return frames


def peak_normalize(frames, target_peak=TARGET_PEAK):
    """Per-window preprocessing: remove DC offset, scale the peak to target_peak.

    This mirrors perch-hoplite's normalize_audio with its default target_peak=0.25.
    Perch V2 is NOT amplitude-invariant, so this shifts the scores — and because every
    threshold below is solved from those scores, skipping it here while using it in
    species_ID.ipynb (or the reverse) biases the whole calibration.

    target_peak=None returns the audio untouched (hoplite's way of disabling this).
    Silent windows (peak 0) are left at zero.
    """
    if target_peak is None:
        return frames.astype(np.float32)
    frames = frames.astype(np.float32).copy()
    frames -= frames.mean(axis=-1, keepdims=True)
    peak = np.max(np.abs(frames), axis=-1, keepdims=True)
    frames = np.divide(frames, peak, out=np.zeros_like(frames), where=peak > 0.0)
    return frames * target_peak

## 4 · Choose the run to calibrate & your validated clips

- **`DETECTIONS_CSV`** — the `all_detections.csv` that `species_ID.ipynb` wrote for the
  run these clips came from. That one file identifies the run, so **`OUTPUT_DIR` derives
  from it**: the thresholds land beside the detections they calibrate, which is also what
  keeps a `thresholds.csv` from drifting away from the run it describes. Override it if
  you would rather collect results elsewhere.
- **`VALID_DIR`** — a folder of clips you have already listened to and sorted. Kept
  separate on purpose: you should copy clips *out* of `segments/` before sorting them,
  since re-running `species_ID.ipynb`'s extraction overwrites that folder.
- **`OUTPUT_DIR`** — where the tables, plot, and report are written (created
  automatically).

If `species_ID.ipynb`'s `manifest.json` sits above that CSV, §7 reads it and **checks its
preprocessing against §2**, so a mismatched `RESAMPLER` or `TARGET_PEAK` is reported
rather than left for you to remember.

Two layouts are accepted.

**One species** — `correct/` and `incorrect/` directly inside `VALID_DIR` (then set
`TARGET_SPECIES` in §5 to that bird's scientific name):

```
VALID_DIR/
  correct/     AT05_20250615_054500_Acrocephalus scirpaceus_0.62_15s.wav
  incorrect/   ...
```

**Several species** — one folder per scientific name, each with its own `correct/`
and `incorrect/` (leave `TARGET_SPECIES = None`). This mirrors the per-species
folders `species_ID.ipynb` writes, so it is the layout you get by sorting each
species folder in place:

```
VALID_DIR/
  Acrocephalus scirpaceus/{correct,incorrect}/...
  Cettia cetti/{correct,incorrect}/...
```

Verdict folder names are case-insensitive (`correct` / `present` / `true` / `tp` / `yes`
= correct; `incorrect` / `absent` / `false` / `fp` / `no` = incorrect).

Species folders must be named for the species, but **you do not have to rename anything**:
`species_ID.ipynb` writes `Aegolius_funereus` (any character outside `[\w.-]` becomes `_`)
and §6 resolves that back to Perch's own `Aegolius funereus`. So the intended workflow is
to sort the folders it produced, in place. A folder that resolves to no Perch class is
reported and skipped, not silently dropped.

One traversal caveat: `rglob` does **not** follow symlinked *sub-directories*. If you
wire in clips from elsewhere, symlink the individual files (or the whole `VALID_DIR`),
not an intermediate folder.

In [ ]:
# Anchor paths to the project folder (the one containing `data/`), walking up from
# wherever Jupyter was launched. VS Code starts the kernel in the notebook's own
# folder, a terminal `jupyter lab` starts it wherever you ran it — this works for both.
PROJECT_DIR = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir()),
    Path.cwd(),
)

# The species_ID.ipynb run being calibrated: its all_detections.csv holds the exact
# confidences (see CONFIDENCE_SOURCE in §5), so nothing has to be re-scored.
DETECTIONS_CSV = (PROJECT_DIR / "data" / "outputs"          # <-- change me: the run to calibrate
                  / "threshold_0.10" / "all_detections.csv")

VALID_DIR  = PROJECT_DIR / "data" / "validated"   # <-- change me: your sorted clips
OUTPUT_DIR = DETECTIONS_CSV.parent / "optimal_threshold"   # results land beside the run
                                                           # they calibrate; override freely

print("Project folder:", PROJECT_DIR)

AUDIO_EXTS = {".wav", ".flac", ".ogg", ".mp3", ".m4a", ".aiff", ".aif"}

def discover_audio(root):
    """Return sorted audio files anywhere under `root`."""
    return sorted(p for p in Path(root).rglob("*") if p.suffix.lower() in AUDIO_EXTS)

Confirm the clips are found, and that the folder layout looks the way you expect, before running the model.

In [ ]:
if not VALID_DIR.exists():
    print(f"⚠ Validated dataset folder not found: {VALID_DIR.resolve()}")
    print("  Point VALID_DIR at a real folder of sorted clips and re-run this cell.")
else:
    found = discover_audio(VALID_DIR)
    print(f"Found {len(found)} audio file(s) under {VALID_DIR.resolve()}\n")
    for sub in sorted(p for p in VALID_DIR.iterdir() if p.is_dir()):
        n = len(discover_audio(sub))
        kids = sorted(d.name for d in sub.iterdir() if d.is_dir())
        print(f"  {sub.name}/  — {n} clip(s)" + (f"  subfolders: {kids}" if kids else ""))

## 5 · Options

- **`CONFIDENCE_SOURCE`** — where each clip's confidence comes from. This is the most
  consequential option here, so it is worth a paragraph:
  - `"detections_csv"` (**recommended**) reads the exact score `species_ID.ipynb` wrote
    to `all_detections.csv`, joining on recording + species + start time.
  - `"filename"` parses it from the clip's own name (`..._0.660_2925s.wav`) — the same
    number, rounded to 3 decimals. Use it when the CSV is gone.
  - `"rescore"` runs Perch over the clip audio again.

  Prefer the first two. `"rescore"` remains an **approximation** — the clip went through
  `librosa.load` a second time when it was cut, so any resampler difference there lands
  it in a slightly different domain (~0.01 on 16 kHz audio). It is also the only slow
  option. Use it for clips with no CSV and no provenance in their name.

- **`RESCORE_WINDOW`** — which 5 seconds of the clip get analysed, on the `"rescore"`
  path only. **The context wings are for listening; the analysis is still one 5 s
  window** — this option decides *which*.
  - `"center"` (**default**) takes the middle 5 s. `extract_segments` centres the
    detection in the clip it writes, so with `CONTEXT_S` wings the detection's own
    window is the centre one.
  - `"best"` frames the clip with `HOP_S` and keeps the highest-scoring window. Use it
    for clips that are *not* centred on their detection.

  This matters more than it sounds. A 5 s detection cut with 1 s wings is a 7 s clip in
  which the detection sits at **1–6 s**, but framing from 0 with `HOP_S = 5.0` yields
  exactly one window, **0–5 s** — the wrong five seconds. Measured against the recorded
  confidences on real clips, mean error **0.093 framing from 0 vs 0.025 centred**, and up
  to 0.23 on a single clip. `"best"` with `HOP_S = 5.0` is the worst case; §7 warns you.

- **`TARGET_SPECIES`** — the scientific name for the one-species layout. Leave `None`
  for the multi-species layout (each subfolder names its own species).
- **`TARGET_PROBABILITY`** — the probability of being correct that the threshold is
  solved for. 0.95 is the convention from the literature.
- **`BIN_EDGES`** — lower edges of the confidence bands in the precision table.
- **`ACTIVATION`** — `"softmax"` (the default, and what Perch was trained with) turns
  the logits into probabilities across all species, exactly as `species_ID.ipynb` does.
  `"sigmoid"` scores each class independently; Perch's winning logits are large, so it
  saturates near 1.0 and is not recommended.
- **`MIN_SAMPLES`** — fewest validated clips before a species is fitted at all.
- **`HOP_S`** — step between the 5 s windows within a clip. Clips are usually about one
  window long, so this rarely matters; 5.0 gives back-to-back windows.
- **`INFER_BATCH`** — how many windows go to the model at once. Lower it if you run out
  of memory. Clips are short, so this matters far less than it does in `species_ID.ipynb`.

In [ ]:
CONFIDENCE_SOURCE  = "detections_csv"   # "detections_csv" | "filename" | "rescore"
RESCORE_WINDOW     = "center"   # "rescore" only: which 5 s to analyse — "center" | "best"

TARGET_SPECIES     = None       # scientific name for the one-species layout; None = one per subfolder
TARGET_PROBABILITY = 0.95       # probability of "correct" the threshold is solved for
BIN_EDGES          = [0.1, 0.3, 0.5]   # lower edges of the precision-table confidence bands
ACTIVATION         = "softmax"  # "softmax" (default, matches Perch's training) or "sigmoid" (per-class)
MIN_SAMPLES        = 8          # fewest validated clips needed before a species is fitted

HOP_S       = 5.0   # step between 5 s windows within a clip (s)
INFER_BATCH = 16    # windows sent to the model per batch; lower this if memory is tight

## 6 · Load the validated clips

Walk `VALID_DIR`, read each clip's verdict from the folder it sits in, and preview how
many correct/incorrect clips were found per species.

In [ ]:
CORRECT_NAMES   = {"correct", "present", "true", "tp", "1", "yes", "positive"}
INCORRECT_NAMES = {"incorrect", "absent", "false", "fp", "0", "no", "negative"}

# species_ID.ipynb names its folders with any character outside [\w.-] replaced by "_",
# so 'Aegolius funereus' arrives here as 'Aegolius_funereus'. Sorting those folders in
# place is the intended workflow, so resolve the folder name back to Perch's own spelling
# instead of making you rename 400 directories by hand.
_NORMALIZED = {re.sub(r"[^a-z]+", " ", n.lower()).strip(): n for n in CLASS_NAMES}


def resolve_species(name):
    """Map a folder name to Perch's spelling of that species, or None if unknown."""
    if name in CLASS_INDEX:
        return name
    spaced = name.replace("_", " ")
    if spaced in CLASS_INDEX:
        return spaced
    return _NORMALIZED.get(re.sub(r"[^a-z]+", " ", name.lower()).strip())


def _verdict(name):
    """Map a folder name to True (correct) / False (incorrect) / None (neither)."""
    key = name.strip().lower()
    if key in CORRECT_NAMES:
        return True
    if key in INCORRECT_NAMES:
        return False
    return None


def _has_verdict_subdirs(directory):
    """True if `directory` has at least one correct/ or incorrect/ subfolder."""
    return any(_verdict(d.name) is not None for d in Path(directory).iterdir() if d.is_dir())


def _collect_species(species_dir, species):
    """Every validated clip under one species folder, tagged correct/incorrect."""
    found = []
    for sub in sorted(Path(species_dir).iterdir()):
        verdict = _verdict(sub.name) if sub.is_dir() else None
        if verdict is None:
            continue
        for path in discover_audio(sub):          # discover_audio defined in §4
            found.append({"path": path, "species": species, "correct": verdict})
    return found


def load_validated_dataset(root, species=None):
    """Discover validated clips under `root` (single- or multi-species layout)."""
    root = Path(root)
    if not root.is_dir():
        raise SystemExit(f"Validated dataset folder not found: {root}")
    if _has_verdict_subdirs(root):                # one-species layout
        if not species:
            raise SystemExit("Found correct/incorrect at the top level; set TARGET_SPECIES.")
        resolved = resolve_species(species)
        if resolved is None:
            raise SystemExit(f"TARGET_SPECIES {species!r} is not a Perch class.")
        files = _collect_species(root, resolved)
    else:                                         # multi-species layout
        files = []
        for species_dir in sorted(p for p in root.iterdir() if p.is_dir()):
            if not _has_verdict_subdirs(species_dir):
                print(f"  skipping '{species_dir.name}': no correct/incorrect subfolders")
                continue
            resolved = resolve_species(species or species_dir.name)
            if resolved is None:
                print(f"  skipping '{species_dir.name}': not a Perch class "
                      "(folder must be named for the species, as species_ID.ipynb writes it)")
                continue
            if resolved != species_dir.name:
                print(f"  '{species_dir.name}' -> '{resolved}'")
            files += _collect_species(species_dir, resolved)
    if not files:
        raise SystemExit(f"No validated clips found under {root} (need correct/ and incorrect/).")
    return files


validated = load_validated_dataset(VALID_DIR, TARGET_SPECIES)
val_df = pd.DataFrame([{"species": f["species"], "correct": f["correct"]} for f in validated])
print(f"Loaded {len(validated)} validated clips across {val_df['species'].nunique()} species:")
print(val_df.groupby("species")["correct"].agg(n="size", correct="sum"))

## 7 · Get each clip's confidence

`CONFIDENCE_SOURCE` decides where the number comes from.

**`"detections_csv"` / `"filename"`** recover the score `species_ID.ipynb` already
computed. Clips are named
`<recording>_<species>_<confidence>_<start>s.wav`, so the recording, the offset and the
confidence can be read straight back out; the CSV is preferred because it carries full
precision rather than 3 decimals, and because a join that misses is loud. Nothing goes
through the model, so this is near-instant.

**`"rescore"`** runs Perch over the clip (reusing §3). The context wings are only there
so you have something to listen to — **the analysis is still a single 5 s window**, and
`RESCORE_WINDOW` (§5) picks it: `"center"` by default, because `extract_segments` centres
the detection in the clip. `ACTIVATION` picks softmax (default) or per-class sigmoid.
This is the slow path, and the only one that can disagree with the detections table the
threshold gets applied to — see §5 for how much and why. The cell warns when the geometry
would silently score the wrong five seconds.

Either way, any clip whose confidence cannot be recovered is **reported and dropped**
rather than guessed at.

In [ ]:
def window_scores(frames):
    """Raw Perch logits for the windows, scored in batches of INFER_BATCH."""
    out = []
    for start in range(0, len(frames), INFER_BATCH):
        batch = frames[start:start + INFER_BATCH]
        out.append(infer(inputs=tf.constant(batch, dtype=tf.float32))["label"].numpy())
    return np.concatenate(out, axis=0)


def center_window(audio):
    """The middle 5 s of a clip, shaped [1, 160000].

    extract_segments centres the detection in the clip it writes, so with CONTEXT_S
    wings the detection's own window is the centre one — NOT the one starting at 0.
    The wings exist to give you something to listen to; the analysis is still 5 s.
    """
    if len(audio) <= WINDOW_SAMPLES:
        return librosa.util.pad_center(audio, size=WINDOW_SAMPLES, axis=-1)[None, :]
    off = (len(audio) - WINDOW_SAMPLES) // 2
    return audio[off:off + WINDOW_SAMPLES][None, :]


def clip_target_confidence(path, species_idx, activation):
    """Perch's confidence for one species on a clip (see RESCORE_WINDOW in §5)."""
    audio = load_audio(path)
    frames = center_window(audio) if RESCORE_WINDOW == "center" else frame_windows(audio, HOP_S)
    frames = peak_normalize(frames)                 # <-- same pipeline as species_ID
    logits = window_scores(frames)
    if activation == "softmax":
        conf = tf.nn.softmax(logits, axis=-1).numpy()[:, species_idx]
    else:  # per-class sigmoid
        conf = 1.0 / (1.0 + np.exp(-np.clip(logits[:, species_idx], -700.0, 700.0)))
    return float(np.max(conf))


# --- recovering the score species_ID.ipynb already computed ------------------
# Its clips are named f'{stem}_{species}_{confidence:.3f}_{int(start_s)}s.wav', with the
# species sanitised the same way its folders are. Anchoring the regex at the END is what
# makes this unambiguous: both the recording stem and the species contain underscores,
# but the confidence and offset do not.
CLIP_NAME_RE = re.compile(r"^(?P<rest>.+)_(?P<conf>\d\.\d+)_(?P<start>\d+)s$")


def sanitize_species(name):
    """Reproduce species_ID.ipynb's folder/filename sanitisation."""
    return re.sub(r"[^\w.-]+", "_", str(name)).strip("_") or "unknown"


def parse_clip_name(path, species):
    """-> (recording_stem, start_s, confidence) from a species_ID clip name, or None."""
    m = CLIP_NAME_RE.match(Path(path).stem)
    if not m:
        return None
    suffix = "_" + sanitize_species(species)
    rest = m["rest"]
    stem = rest[:-len(suffix)] if rest.endswith(suffix) else rest
    return stem, int(m["start"]), float(m["conf"])


def load_detection_index(csv_path):
    """(recording stem, sanitised species, start_s) -> confidence, from a detections CSV."""
    df = pd.read_csv(csv_path)
    missing = {"file", "label", "start_s", "confidence"} - set(df.columns)
    if missing:
        raise SystemExit(f"{csv_path} is missing column(s): {sorted(missing)}")
    return {
        (Path(r.file).stem, sanitize_species(r.label), int(r.start_s)): float(r.confidence)
        for r in df.itertuples(index=False)
    }


# --- collect one confidence per validated clip -------------------------------
detection_index = {}
if CONFIDENCE_SOURCE == "rescore" and validated:
    if RESCORE_WINDOW not in {"center", "best"}:
        raise SystemExit(f"Unknown RESCORE_WINDOW: {RESCORE_WINDOW!r}")
    # A clip cut with CONTEXT_S wings is longer than one window and its detection does not
    # start at 0, so framing from 0 with a full-window hop scores the wrong 5 s. See §5.
    clip_s = librosa.get_duration(path=str(validated[0]["path"]))
    if clip_s > WINDOW_S + 1e-3 and RESCORE_WINDOW == "best" and HOP_S >= WINDOW_S:
        print(f"⚠ Clips are {clip_s:.1f} s but HOP_S = {HOP_S} gives a single window "
              f"(0–{WINDOW_S:.0f} s).\n"
              "  If these were cut with CONTEXT_S > 0 the detection is NOT at the start, so\n"
              '  this scores the wrong window. Use RESCORE_WINDOW = "center", lower HOP_S,\n'
              '  or better, CONFIDENCE_SOURCE = "detections_csv".\n')
if CONFIDENCE_SOURCE == "detections_csv":
    if not Path(DETECTIONS_CSV).is_file():
        found = sorted((PROJECT_DIR / "data" / "outputs").rglob("all_detections.csv"))
        hint = ("\n  Found these under data/outputs — set DETECTIONS_CSV to one of them:\n"
                + "\n".join(f"    {p}" for p in found[:10])) if found else                "\n  No all_detections.csv found under data/outputs — run species_ID.ipynb first."
        raise SystemExit(f"DETECTIONS_CSV not found: {DETECTIONS_CSV}{hint}\n"
                         '  Or set CONFIDENCE_SOURCE = "filename" to read it from the clip names.')
    detection_index = load_detection_index(DETECTIONS_CSV)
    print(f"Loaded {len(detection_index)} detections from {DETECTIONS_CSV}")

    # species_ID.ipynb writes manifest.json above its threshold_XX/ folder. If it is
    # there, check that run's preprocessing against this one instead of trusting that
    # you remembered: a threshold is only valid for the pipeline that produced the scores.
    # Only the CSV's own folder and the one above it: species_ID writes manifest.json
    # exactly one level up from threshold_XX/. Walking further would happily pick up an
    # unrelated run's manifest from higher in data/outputs and compare against that.
    run_manifest = next((d / "manifest.json" for d in list(Path(DETECTIONS_CSV).parents)[:2]
                         if (d / "manifest.json").is_file()), None)
    if run_manifest is None:
        print("  (no manifest.json beside the run — cannot verify its preprocessing)")
    else:
        ran = json.loads(run_manifest.read_text())
        differs = {k: (ran.get(k), mine) for k, mine in
                   [("resampler", RESAMPLER), ("target_peak", TARGET_PEAK),
                    ("sample_rate", SAMPLE_RATE), ("window_s", WINDOW_S)]
                   if k in ran and ran.get(k) != mine}
        if differs:
            print(f"\n⚠ {run_manifest} disagrees with this notebook:")
            for k, (theirs, mine) in differs.items():
                print(f"    {k}: run used {theirs!r}, §2 here says {mine!r}")
            print("  On CONFIDENCE_SOURCE = 'detections_csv' the scores come from that run,\n"
                  "  so the thresholds describe ITS pipeline — align §2 before reporting them.\n")
        else:
            print(f"  preprocessing matches the run ({run_manifest.name}): "
                  f"resampler={RESAMPLER}, target_peak={TARGET_PEAK}")
elif CONFIDENCE_SOURCE not in {"filename", "rescore"}:
    raise SystemExit(f'Unknown CONFIDENCE_SOURCE: {CONFIDENCE_SOURCE!r}')

scores_by_species = {}          # species -> {"confidence": [...], "correct": [...]}
unresolved = []                 # clips whose confidence could not be recovered
for j, f in enumerate(validated, start=1):
    idx = CLASS_INDEX.get(f["species"])
    if idx is None:
        raise SystemExit(f"'{f['species']}' is not a Perch class; use the scientific name.")

    if CONFIDENCE_SOURCE == "rescore":
        print(f"[{j}/{len(validated)}] {f['species']} · {Path(f['path']).name}")
        confidence = clip_target_confidence(f["path"], idx, ACTIVATION)
    else:
        parsed = parse_clip_name(f["path"], f["species"])
        if parsed is None:
            unresolved.append((f["path"], "filename does not carry a confidence"))
            continue
        stem, start_s, name_conf = parsed
        if CONFIDENCE_SOURCE == "filename":
            confidence = name_conf
        else:
            key = (stem, sanitize_species(f["species"]), start_s)
            if key not in detection_index:
                unresolved.append((f["path"], f"no detection row for {key}"))
                continue
            confidence = detection_index[key]

    bucket = scores_by_species.setdefault(f["species"], {"confidence": [], "correct": []})
    bucket["confidence"].append(confidence)
    bucket["correct"].append(bool(f["correct"]))

print(f"Confidence source: {CONFIDENCE_SOURCE}")
print("Scored:", {s: len(v["correct"]) for s, v in scores_by_species.items()})
if unresolved:
    print(f"\n⚠ Dropped {len(unresolved)} clip(s) with no recoverable confidence:")
    for path, why in unresolved[:10]:
        print(f"    {Path(path).name} — {why}")
    if len(unresolved) > 10:
        print(f"    … and {len(unresolved) - 10} more")
    print('  If these did not come from species_ID.ipynb, use CONFIDENCE_SOURCE = "rescore".')
if not scores_by_species:
    raise SystemExit("No clip yielded a confidence — check CONFIDENCE_SOURCE and DETECTIONS_CSV.")

## 8 · Fit the logistic model per species

The statistics below take plain arrays (confidence + correctness), so they run without
the model — you can re-run this section with different options instantly, without
re-scoring the clips.

`fit_species_threshold` fits `P(correct) = sigmoid(b0 + b1·L)`, solves for the
target-probability threshold, builds the precision bins, and computes the fitted curve
plus its 95 % band. Species with too few clips, all-correct / all-incorrect data, or a
non-positive slope are reported but not fitted — the reason lands in `note`.

In [ ]:
EPS = 1e-6


def logit_score(confidence):
    """Standard logit L = ln(c / (1 - c)) (inverse sigmoid), clipped to stay finite."""
    c = np.clip(np.asarray(confidence, dtype=np.float64), EPS, 1.0 - EPS)
    return np.log(c / (1.0 - c))


def sigmoid(eta):
    """Logistic sigmoid, argument clipped to avoid exp overflow."""
    return 1.0 / (1.0 + np.exp(-np.clip(eta, -700.0, 700.0)))


def precision_bins(confidence, correct, bin_edges):
    """Bin detections by confidence; count verified (correct) per band."""
    conf = np.asarray(confidence, dtype=np.float64)
    ok = np.asarray(correct).astype(bool)
    edges = list(bin_edges) + [1.0 + EPS]
    rows = []
    for i in range(len(edges) - 1):
        low, high = edges[i], edges[i + 1]
        in_bin = (conf >= low) & (conf < high)
        is_last = i == len(edges) - 2
        disp_high = min(high, 1.0)
        det = int(in_bin.sum())
        ver = int((in_bin & ok).sum())
        rows.append({
            "category": f"[{low:.2f}, {disp_high:.2f}{']' if is_last else ')'}",
            "detections": det, "verified": ver,
            "precision": ver / det if det else float("nan"),
        })
    return rows


def _fit_logistic(logit, correct):
    """Unregularised MLE fit; returns (intercept, slope, covariance)."""
    model = LogisticRegression(C=np.inf, solver="lbfgs", max_iter=1000)
    model.fit(logit.reshape(-1, 1), correct.astype(int))
    b0, b1 = float(model.intercept_[0]), float(model.coef_[0][0])
    design = np.column_stack([np.ones_like(logit), logit])
    p = sigmoid(b0 + b1 * logit)
    w = np.clip(p * (1.0 - p), 1e-12, None)
    fisher = design.T @ (design * w[:, None])
    try:
        cov = np.linalg.inv(fisher)
    except np.linalg.LinAlgError:
        cov = np.linalg.pinv(fisher)
    return b0, b1, cov


def fit_species_threshold(species, confidence, correct, target_probability=0.95,
                          bin_edges=(0.1, 0.3, 0.5), min_samples=8):
    """Fit the logistic model and solve for the target-probability threshold."""
    conf = np.clip(np.asarray(confidence, dtype=np.float64), 0.0, 1.0)
    ok = np.asarray(correct).astype(bool)
    n, n_correct = int(conf.size), int(ok.sum())
    n_incorrect = n - n_correct
    result = {
        "species": species, "n": n, "n_correct": n_correct, "n_incorrect": n_incorrect,
        "target_probability": target_probability, "threshold": float("nan"),
        "intercept": float("nan"), "slope": float("nan"), "fitted": False, "note": "",
        "bins": precision_bins(conf, ok, bin_edges),
        "points_conf": conf, "points_correct": ok.astype(float),
        "curve_conf": np.empty(0), "curve_prob": np.empty(0),
        "curve_lo": np.empty(0), "curve_hi": np.empty(0),
    }
    if n < min_samples:
        result["note"] = f"only {n} validated detections (need >= {min_samples})"
        return result
    if n_correct == 0 or n_incorrect == 0:
        only = "correct" if n_incorrect == 0 else "incorrect"
        result["note"] = f"all detections are {only}; cannot fit a regression"
        return result

    logit = logit_score(conf)
    b0, b1, cov = _fit_logistic(logit, ok)
    result["intercept"], result["slope"] = b0, b1
    if b1 <= 0:
        result["note"] = "confidence not positively associated with correctness (slope <= 0)"
        return result

    target_logit = float(np.log(target_probability / (1.0 - target_probability)))
    l_star = (target_logit - b0) / b1
    threshold = float(sigmoid(l_star))              # sigmoid => always in (0, 1)
    result["threshold"], result["fitted"] = threshold, True
    if threshold >= conf.max():
        result["note"] = "threshold exceeds the largest validated confidence (extrapolated)"
    elif threshold <= conf.min():
        result["note"] = "target met below the smallest validated confidence (extrapolated)"

    # Fitted probability curve + 95% band (delta method) for the plot.
    grid = np.linspace(EPS, 1.0 - EPS, 200)
    glogit = logit_score(grid)
    design = np.column_stack([np.ones_like(glogit), glogit])
    eta = b0 + b1 * glogit
    se = np.sqrt(np.clip(np.einsum("ij,jk,ik->i", design, cov, design), 0.0, None))
    result["curve_conf"], result["curve_prob"] = grid, sigmoid(eta)
    result["curve_lo"], result["curve_hi"] = sigmoid(eta - 1.96 * se), sigmoid(eta + 1.96 * se)
    return result


threshold_results = []
for species in sorted(scores_by_species):
    conf = np.asarray(scores_by_species[species]["confidence"], dtype=np.float64)
    correct = np.asarray(scores_by_species[species]["correct"], dtype=bool)
    r = fit_species_threshold(species, conf, correct, target_probability=TARGET_PROBABILITY,
                              bin_edges=BIN_EDGES, min_samples=MIN_SAMPLES)
    threshold_results.append(r)
    if r["fitted"]:
        print(f"{species}: threshold = {r['threshold']:.3f}  (n={r['n']}, {r['n_correct']} correct)"
              + (f"  — {r['note']}" if r["note"] else ""))
    else:
        print(f"{species}: no threshold — {r['note']}")

## 9 · Tables: estimated thresholds & precision by confidence band

Two tables, saved to `OUTPUT_DIR` and shown inline: the per-species thresholds, and the
precision breakdown per confidence band (the totals row lets you sanity-check the fit
against what you actually verified).

A `manifest.json` records the settings — including `resampler` and `target_peak`, so a
threshold can be traced back to the preprocessing it is valid for.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- thresholds table (one row per species) ---------------------------------
thr_rows = [{
    "species": r["species"],
    "threshold": round(r["threshold"], 4) if r["fitted"] else None,
    "target_probability": r["target_probability"],
    "n": r["n"], "n_correct": r["n_correct"], "n_incorrect": r["n_incorrect"],
    "total_precision_pct": round(100 * r["n_correct"] / r["n"], 1) if r["n"] else None,
    "intercept": round(r["intercept"], 4) if r["fitted"] else None,
    "slope": round(r["slope"], 4) if r["fitted"] else None,
    "fitted": r["fitted"], "note": r["note"],
} for r in threshold_results]
thresholds_df = pd.DataFrame(thr_rows)
thresholds_df.to_csv(OUTPUT_DIR / "thresholds.csv", index=False)
(OUTPUT_DIR / "thresholds.json").write_text(json.dumps(thr_rows, indent=2), encoding="utf-8")

# --- precision-by-confidence-band table -------------------------------------
prec_rows = []
for r in threshold_results:
    for b in r["bins"]:
        prec_rows.append({
            "species": r["species"], "confidence_category": b["category"],
            "detections": b["detections"], "verified": b["verified"],
            "precision_pct": round(100 * b["precision"], 1) if b["detections"] else None,
        })
    tot_det = sum(b["detections"] for b in r["bins"])
    tot_ver = sum(b["verified"] for b in r["bins"])
    prec_rows.append({
        "species": r["species"], "confidence_category": "TOTAL",
        "detections": tot_det, "verified": tot_ver,
        "precision_pct": round(100 * tot_ver / tot_det, 1) if tot_det else None,
    })
precision_df = pd.DataFrame(prec_rows)
precision_df.to_csv(OUTPUT_DIR / "precision_table.csv", index=False)

# --- manifest: the settings these thresholds are valid for -------------------
manifest = {
    "model_slug": MODEL_SLUG,
    "sample_rate": SAMPLE_RATE,
    "resampler": RESAMPLER,
    "target_peak": TARGET_PEAK,
    "window_s": WINDOW_S,
    "hop_s": HOP_S,
    "activation": ACTIVATION,
    "confidence_source": CONFIDENCE_SOURCE,
    "rescore_window": RESCORE_WINDOW if CONFIDENCE_SOURCE == "rescore" else None,
    "detections_csv": str(DETECTIONS_CSV) if CONFIDENCE_SOURCE == "detections_csv" else None,
    "n_dropped_clips": len(unresolved),
    "target_probability": TARGET_PROBABILITY,
    "bin_edges": BIN_EDGES,
    "min_samples": MIN_SAMPLES,
    "n_classes": len(CLASS_NAMES),
    "n_clips": len(validated),
    "n_species": len(threshold_results),
}
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(f"Wrote thresholds.csv, thresholds.json, precision_table.csv, manifest.json -> {OUTPUT_DIR}")
display(thresholds_df)
display(precision_df)

## 10 · Plot: probability of a correct detection

One panel per species: the validated clips as a 0/1 scatter (jittered so overlapping
points stay visible), the fitted logistic curve with its 95 % band, the
target-probability line, and the estimated threshold where the curve crosses it.

In [ ]:
def plot_probability_curves(results, path):
    """Per-species panel: 0/1 scatter, fitted curve + 95% band, target & threshold lines."""
    n = max(1, len(results))
    ncols = min(2, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.2 * nrows), squeeze=False)
    rng = np.random.default_rng(0)
    flat = axes.ravel()
    for ax, r in zip(flat, results, strict=False):
        jitter = rng.uniform(-0.03, 0.03, size=r["points_correct"].shape)
        ax.scatter(r["points_conf"], r["points_correct"] + jitter, s=10, color="black",
                   alpha=0.5, zorder=3)
        if r["fitted"] and r["curve_conf"].size:
            ax.plot(r["curve_conf"], r["curve_prob"], color="tab:blue", lw=2, zorder=2)
            ax.fill_between(r["curve_conf"], r["curve_lo"], r["curve_hi"],
                            color="tab:blue", alpha=0.15, zorder=1)
            ax.axhline(r["target_probability"], color="gray", ls=":", lw=1)
            if np.isfinite(r["threshold"]):
                ax.axvline(r["threshold"], color="tab:red", ls="--", lw=1.5, zorder=4)
                ax.annotate(f"threshold = {r['threshold']:.2f}", xy=(r["threshold"], 0.5),
                            xytext=(6, 0), textcoords="offset points", color="tab:red",
                            fontsize=8, rotation=90, va="center")
        ax.set_xlim(0, 1)
        ax.set_ylim(-0.08, 1.08)
        ax.set_xlabel("Confidence score")
        ax.set_ylabel("Prob. correct detection")
        ax.set_title(r["species"] if r["fitted"] else f"{r['species']} (no fit)", fontsize=10)
        ax.grid(True, alpha=0.25)
    for ax in flat[len(results):]:
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    return fig


fig = plot_probability_curves(threshold_results, OUTPUT_DIR / "probability_curves.png")
plt.show()
print(f"Saved plot -> {OUTPUT_DIR / 'probability_curves.png'}")

## 11 · A `report.md`

Bundles both tables and the figure into one human-readable Markdown file, next to the
CSVs — the thing to hand to a collaborator.

In [ ]:
target = threshold_results[0]["target_probability"] if threshold_results else TARGET_PROBABILITY
lines = [
    "# Optimal Confidence Threshold", "",
    f"Confidence threshold for a **{target:.0%} probability of correct identification**, "
    "estimated per species by logistic regression of human-validated detections on the "
    "logit-transformed confidence score.", "",
    f"Confidence source: `{CONFIDENCE_SOURCE}`. Scores produced with resampler "
    f"`{RESAMPLER}` and target peak `{TARGET_PEAK}` (activation: `{ACTIVATION}`) — the "
    "thresholds below are valid for that pipeline.", "",
    "## Estimated thresholds", "",
    "| Species | Threshold | n | Correct | Incorrect | Overall precision |",
    "| --- | --- | --- | --- | --- | --- |",
]
for r in threshold_results:
    thr = f"**{r['threshold']:.2f}**" if r["fitted"] else "—"
    prec = f"{100 * r['n_correct'] / r['n']:.1f}%" if r["n"] else "—"
    lines.append(f"| {r['species']} | {thr} | {r['n']} | {r['n_correct']} | {r['n_incorrect']} | {prec} |")
    if r["note"]:
        lines.append(f"|   ↳ *{r['note']}* | | | | | |")
lines += ["", "## Precision by confidence category", ""]
for r in threshold_results:
    lines += [f"### {r['species']}", "",
              "| Confidence category | Detections | Verified | Precision |",
              "| --- | --- | --- | --- |"]
    for b in r["bins"]:
        prec = f"{100 * b['precision']:.1f}%" if b["detections"] else "—"
        lines.append(f"| {b['category']} | {b['detections']} | {b['verified']} | {prec} |")
    tot_det = sum(b["detections"] for b in r["bins"])
    tot_ver = sum(b["verified"] for b in r["bins"])
    tot_prec = f"{100 * tot_ver / tot_det:.1f}%" if tot_det else "—"
    lines += [f"| **TOTAL** | {tot_det} | {tot_ver} | {tot_prec} |", ""]
lines += ["## Figure", "", "![Probability of correct detection](probability_curves.png)", ""]
(OUTPUT_DIR / "report.md").write_text("\n".join(lines), encoding="utf-8")
print(f"Wrote report.md -> {OUTPUT_DIR / 'report.md'}")

## 12 · Using the thresholds

`thresholds.csv` now holds one confidence cutoff per species. Back in `species_ID.ipynb`,
a detection of *Acrocephalus scirpaceus* below that species' threshold is one you should
not report without listening to it — and one above it is correct about 95 % of the time
(or whatever `TARGET_PROBABILITY` you set).

The practical recipe: run `species_ID.ipynb` at a **low** `THRESHOLD` (0.1, say) so
nothing is thrown away early, then filter its `all_detections.csv` per species using this
table. A single global cutoff cannot do that — it is either too strict for the species
Perch finds hard or too loose for the ones it finds easy.

A threshold is only as good as the clips it was fitted on: it applies to recordings like
the ones you validated (same site, season, recorder, noise floor) **and** to scores
produced by the same preprocessing (§2's `RESAMPLER` and `TARGET_PEAK`). Re-fit when the
dataset changes.